# HVG sweep — build the 1k / 2k / 3k variants

Pipeline driver living in `analysis/`, which is why it does not look like a dependency until you read
it: `verify_variants` §9 plots performance against gene-set size and needs these three variants to
exist.

| | |
|---|---|
| **reads** | the same raw sources `1_data` reads |
| **writes** | `<hvg1000\|hvg2000\|hvg3000>/` — the full four-file chain per variant |

**Why it drives `pipeline.*` rather than reimplementing anything.** The sweep's points must be built
exactly as `hvg5000` and `all_genes` are, or the curve's x-axis confounds gene count with code
version. The score and the component count are passed explicitly and checked against the chain's
defaults, so a default drifting raises instead of silently producing a different variant.

⚠️ **Expensive** — three scGPT embeddings. Gated behind `RUN_HVG_SWEEP`, which is the only switch:
`True` means rebuild, including replacing artifacts that already exist.

In [1]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import inspect

from scripts.layout import DEFAULT_CTRP_SCORE, PipelinePaths
from scripts.preprocessing import add_pca, pipeline

# ⚠️ Was True for the 13.08.2026 re-embed (Selin), which REVERSED R1's hvg5000 + all_genes choice
# because verify_variants §9 plots performance against gene-set size and, with only two variants
# rebuilt, its five points differed in gene count AND in code version so no trend could be read off
# the curve. That rebuild happened: all three variants exist, built 13.08.2026 15:44-16:38, and the
# sweep is like-for-like (steps/05, "The gene-set sweep, on live numbers").
#
# ✅ SET BACK TO False, 14.08.2026 — this cell's own instruction above, now that the sweep exists.
# Unlike 4a's section flags, which stay True because that notebook INTERPRETS the artifacts it
# regenerates, this one only BUILDS ~11 GB of intermediate h5ads that other notebooks consume. Left
# True it would rebuild three scGPT embeddings (~54 min) on every execution and overwrite the inputs
# the committed gene-set sweep was measured on. Flip it to True to rebuild them.
RUN_HVG_SWEEP = False
SCGPT_PYTHON = '/Users/selin/PycharmProjects/scGPT/.venv/bin/python'
SWEEP_VARIANTS = ['hvg1000', 'hvg2000', 'hvg3000']
PCA_N_COMPS = add_pca.DEFAULT_N_COMPS      # 512, matching the scGPT width

# THIS NOTEBOOK MUST BUILD ITS VARIANTS EXACTLY AS THE NUMBERED CHAIN BUILDS hvg5000 AND all_genes.
# `verify_variants` §9 plots performance against gene-set size across all five, so if the two paths
# construct their variants differently the curve's points differ in construction AS WELL AS in gene
# count -- and no trend can be read off it. That is precisely the defect this rebuild exists to
# remove, so it would be absurd to reintroduce it here.
#
# Until 13.08.2026 the two paths agreed only BY COINCIDENCE: this notebook omitted the score (taking
# `DEFAULT_CTRP_SCORE`, which happened to equal the chain's `SCORE`) and passed `PCA_N_COMPS`, which
# happened to equal `pipeline.pca`'s own default. Change either default and the sweep would silently
# start building variants the chain does not -- with nothing raising. The agreement is now stated and
# asserted instead of inherited.
SCORE = DEFAULT_CTRP_SCORE                 # the chain's 1_data / 3_representations pass this
_pca_default = inspect.signature(pipeline.pca).parameters['n_comps'].default
if PCA_N_COMPS != _pca_default:
    raise ValueError(
        f'PCA_N_COMPS={PCA_N_COMPS} but pipeline.pca defaults to {_pca_default}, which is what the '
        f'numbered chain uses. The sweep would build its variants on a different number of '
        f'components than hvg5000 and all_genes, and verify_variants §9 would compare them as if '
        f'only the gene count differed. Pass the same value in both places, or change both.')

print(f'score={SCORE}  n_comps={PCA_N_COMPS} (matches pipeline.pca default)')
for v in SWEEP_VARIANTS:
    p = PipelinePaths.build(None, v, SCORE)
    print(f'  {v:9s} {"present" if p.targets_h5ad.exists() else "MISSING"}')

score=auc_cc  n_comps=512 (matches pipeline.pca default)
  hvg1000   present
  hvg2000   present
  hvg3000   present


## Build

| | |
|---|---|
| **out** | per variant: raw h5ad → embeddings → targets → splits → PCA |

The same six steps in the same order as the numbered chain, once per variant. `overwrite=True` on the
two guarded steps, because replacing the existing artifacts is the point of running this.

In [2]:
if not RUN_HVG_SWEEP:
    print('RUN_HVG_SWEEP is False -- skipping the heavy scGPT sweep build.')
else:
    for variant in SWEEP_VARIANTS:
        paths = PipelinePaths.build(None, variant, SCORE)
        # ⚠️ The "targets present -> skip" guard is DELIBERATELY GONE (13.08.2026, Selin).
        # It read:
        #     if paths.targets_h5ad.exists():
        #         print(f'{variant}: targets present, skipping.'); continue
        # which made this cell a no-op for exactly the case it is now being run for: all three
        # variants already had targets, built 01:53 on 13.08.2026 by the pre-correction code. With
        # the guard in place, flipping RUN_HVG_SWEEP and passing overwrite=True would still have
        # skipped every variant and reported success -- the same shape of silent no-op this review
        # has been finding all day. RUN_HVG_SWEEP is now the only switch: True means rebuild.
        print('\n' + '=' * 70 + f'\n{variant}\n' + '=' * 70)
        pipeline.fetch(paths)
        # overwrite=True on the two guarded steps, because replacing these artifacts IS the point.
        # convert() attaches total_counts / pct_counts_mt itself, so the rebuilt variants carry the
        # UMI covariates the 01:53 ones lack -- which is half of what made the sweep unreadable.
        pipeline.convert(paths, overwrite=True)
        pipeline.scgpt(paths, SCGPT_PYTHON, overwrite=True)
        pipeline.targets(paths)
        pipeline.splits(paths)
        pipeline.pca(paths, n_comps=PCA_N_COMPS)
        print(f'{variant}: built -> {paths.targets_h5ad}')

RUN_HVG_SWEEP is False -- skipping the heavy scGPT sweep build.
